# T5-Small — DIMER text-to-text generation tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/t5-small-text2text-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/t5-small-text2text-pipeline/blob/main/tutorials/t5_small_text2text_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--t5%2Ft5--small-ffcc4d?style=flat)](https://huggingface.co/google-t5/t5-small)
[![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftext--to--text--transfer--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/text-to-text-transfer-transformer)
[![arXiv](https://img.shields.io/badge/arXiv-1910.10683-b31b1b.svg)](https://arxiv.org/abs/1910.10683)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** caller-prefixed text-to-text generation (summarisation and English→German/French/Romanian translation) using the pinned `google-t5/t5-small` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`T5SmallText2TextPipeline`) rather than reimplementing model inference. `google-t5/t5-small` is the original 60 M-parameter T5 of Raffel et al. (2020) — **not FLAN-T5**, so it follows only the task prefixes it was trained on, not free-form instructions. At inference the encoder reads the whole prefixed input once and the decoder emits one SentencePiece token per step until `</s>` or a step ceiling; **greedy decoding** (per-step argmax, `num_beams=1`, `do_sample=False`) is the default decision rule and beam search is available on request — there is no sampling and no seed. The caller writes the task prefix — `summarize: `, `translate English to German: `, `translate English to French: ` or `translate English to Romanian: ` — in front of the text: **the pipeline invents no prefix** and only reports which known one an input started with (`known_prefix`, or `None`). **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the model, the SentencePiece tokenizer and the prefix convention; what this repository adds is manifest verification, input validation with named ceilings, a fixed output contract, and the list of trained prefixes as `TASK_PREFIXES`.

**Learning objectives:** bootstrap the repository in a fresh runtime, author two already-prefixed inputs (or upload your own), surface the pipeline's ceilings and the decision rule before any model work, stage and digest-verify the immutable upstream snapshot through the package, generate through the public API with explicit `max_new_tokens`/`num_beams`, read `stopped_by`, `known_prefix` and the token counts correctly, understand why **no metric is reported** and what references a ROUGE/BLEU evaluation would need, and export every generation with its identifier plus provenance.

**This notebook does not demonstrate:** instruction following or chat, sampling-based decoding, batching, classification or embedding (the GLUE tasks in the training mixture are not exposed), source languages other than English or target languages other than German, French and Romanian, the upstream `task_specific_params` decoding settings (`min_length`, `length_penalty`, `no_repeat_ngram_size` are not applied by the pipeline), or any ROUGE/BLEU measurement. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float32 there too). CPU is adequate: the repository's model card records, for the Windows-venv smoke on an Intel Core Ultra 9 275HX, 3.89 s to load and digest-verify the 244 MB snapshot, 0.16 s for the 11-token translation (greedy; 0.09 s with `num_beams=4`) and 0.31 s for a 94-token `summarize: ` input that generated 48 tokens; the `t5-base-text2text-pipeline` sibling (220 M parameters, 892 MB snapshot) took 1.2× as long to load and 1.8–2.7× as long per call on the same CPU in its own smoke, as both cards record. The pinned `torch==2.14.0` install and the 242 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what greedy decoding and beam search do; why a generated sentence can be fluent and wrong.
- **Data:** the default sample is **synthetic** — two already-prefixed inputs authored in code (one translation sentence, one four-sentence passage to summarise) — so nothing is downloaded and no private data is needed. It carries no reference outputs, so any number it produces is smoke/sanity evidence, never a quality measurement. BYOD is one UTF-8 text file, gated off by default; each non-empty line is one input that must already start with its task prefix. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** GitHub (repository clone), PyPI (pinned wheels) and the Hugging Face Hub (the package's `stage_missing_files` fetches the git-ignored `model.safetensors` once at the immutable revision, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `tokenizers`, `sentencepiece`, `huggingface-hub`, `safetensors`, `numpy`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference is float32 on CPU and on CUDA; no compilation, quantization or FlashAttention is applied. Look for a dictionary reporting the repository revision, Python, `torch` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/t5-small-text2text-pipeline.git'
REPO_NAME = 't5-small-text2text-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: two inputs authored in this cell, each already carrying its task prefix because the pipeline invents no prefix. The first is the translation sentence the repository's card-pass smoke used (`translate English to German: The house is wonderful.`); the second is a four-sentence English passage about the C4 corpus, written here after the T5 paper's description, behind `summarize: `. Neither has a reference output — no human translation, no reference summary — so nothing in this notebook is a quality measurement; the card's smoke observation that the translation came back as `Das Haus ist wunderbar.` is one run on one machine, not an expected value this notebook asserts. The sample identity and a SHA-256 of its text are printed so an export can be tied to exactly these inputs.

Two Colab form parameters fix the generation settings for every call: `GEN_MAX_NEW_TOKENS` (default 64, the package's `DEFAULT_MAX_NEW_TOKENS`) and `NUM_BEAMS` (default 1 = greedy). They are checked against the package ceilings in the next section.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file in which every non-empty line is one input **that already starts with its task prefix** (for example `translate English to French: ...`); each line must be at most `MAX_TEXT_CHARS` characters and tokenise to at most `MAX_INPUT_TOKENS` SentencePiece pieces, which the pipeline enforces by rejecting, not by truncating. The upload stays inside this runtime. If you hold reference outputs for your lines, keep them outside the notebook — Section 5 explains what to compute with them.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
GEN_MAX_NEW_TOKENS = 64  # @param {type:"integer"}
NUM_BEAMS = 1  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    texts = [line.strip() for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if not texts:
        raise ValueError(f'{sample_name}: expected at least one non-empty line, each starting with its task prefix')
    sample_kind = 'BYOD upload'
else:
    texts = [
        'translate English to German: The house is wonderful.',
        'summarize: The Colossal Clean Crawled Corpus, or C4, is a cleaned version of the April 2019 Common Crawl web snapshot. '
        'The corpus was prepared by keeping only lines that end in terminal punctuation, dropping pages with fewer than five sentences, '
        'and removing pages that contain words from a block list, placeholder text or source code. '
        'Duplicate three-sentence spans were also removed so that boilerplate does not dominate the training signal. '
        'It was used to pre-train the T5 family of models with a span-corruption denoising objective.',
    ]
    sample_name = 'synthetic_prefixed_inputs'
    sample_kind = 'synthetic (authored in this cell)'
item_ids = [f'input{index:02d}' for index in range(len(texts))]
sample_sha256 = hashlib.sha256('\n'.join(texts).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256, 'max_new_tokens': GEN_MAX_NEW_TOKENS, 'num_beams': NUM_BEAMS})
for item_id, text in zip(item_ids, texts):
    print(f'{item_id}: {text[:110]}{"..." if len(text) > 110 else ""}')

## 3. Surface the ceilings and the decision rule, then check the inputs

The pipeline enforces named ceilings, imported here from the package so the values shown are the ones in force: `MAX_TEXT_CHARS` (characters per input, checked before tokenisation), `MAX_INPUT_TOKENS` (encoder tokens including `</s>`; the upstream `n_positions` — **longer inputs are rejected with a `ValueError` naming the count, never silently cut**), `MAX_NEW_TOKENS` (ceiling on `max_new_tokens`), `MAX_NUM_BEAMS` (ceiling on `num_beams`) and `DEFAULT_MAX_NEW_TOKENS`. `DECISION_RULE` states the decoding rule in force (greedy per-step argmax, beam search when `num_beams > 1`, no sampling) and `TASK_PREFIXES` lists the four prefixes the checkpoint was trained on. This cell checks what can be checked before the tokenizer exists — non-empty text, the character ceiling, the two generation settings, and whether each input starts with a known prefix (an unknown prefix is **not** rejected, because the pipeline does not refuse it either, but it is flagged: the model then returns something plausible-looking rather than an error). The token ceiling is enforced by the pipeline itself with the real tokenizer inside `generate`, and every result reports `input_tokens`. Nothing here trims or alters the texts.

In [ ]:
from t5_small_text2text_pipeline import DECISION_RULE, DEFAULT_MAX_NEW_TOKENS, MAX_INPUT_TOKENS, MAX_NEW_TOKENS, MAX_NUM_BEAMS, MAX_TEXT_CHARS, TASK_PREFIXES

ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS}
print(ceilings)
print({'decision_rule': DECISION_RULE})
print({'task_prefixes': list(TASK_PREFIXES)})
problems = []
if not 1 <= GEN_MAX_NEW_TOKENS <= MAX_NEW_TOKENS:
    problems.append(f'GEN_MAX_NEW_TOKENS={GEN_MAX_NEW_TOKENS} is outside 1..MAX_NEW_TOKENS={MAX_NEW_TOKENS}: set it within range')
if not 1 <= NUM_BEAMS <= MAX_NUM_BEAMS:
    problems.append(f'NUM_BEAMS={NUM_BEAMS} is outside 1..MAX_NUM_BEAMS={MAX_NUM_BEAMS}: set it within range')
for item_id, text in zip(item_ids, texts):
    if not text.strip():
        problems.append(f'{item_id} is empty: remove blank lines from the input')
    if len(text) > MAX_TEXT_CHARS:
        problems.append(f'{item_id} has {len(text)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten the text')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
known_prefixes = {item_id: next((prefix for prefix in TASK_PREFIXES if text.startswith(prefix)), None) for item_id, text in zip(item_ids, texts)}
unknown = [item_id for item_id, prefix in known_prefixes.items() if prefix is None]
if unknown:
    print({'warning': f'{unknown} start with no trained prefix; the model will still return text, but not a summary or translation'})
print({'inputs': len(texts), 'longest_chars': max(len(text) for text in texts), 'known_prefixes': known_prefixes, 'within_ceilings': True, 'token_ceiling': 'enforced by generate() with the real tokenizer; reported as input_tokens'})

## 4. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here) and refuses remote model code (`trust_remote_code=False`). The Git repository commits the DIMER snapshot manifest (`weights/t5-small/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the 7 snapshot files) and the small files — `config.json`, `generation_config.json`, `spiece.model`, `tokenizer.json`, `tokenizer_config.json`, the upstream `README.md` and `dimer-base-manifest.json` — but git-ignores the 242 MB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, from the Hub at the pinned revision, into the snapshot directory; it returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` on a warm runtime) and refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every listed file (size and SHA-256), raises on the first mismatch, and its returned dict is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the tokenizer and the float32 model from the verified directory with `local_files_only=True` — there is no fallback to a different download. The effective model identity and the device chosen (`cuda:0` when available, else `cpu`) are printed before inference.

In [ ]:
from t5_small_text2text_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, T5SmallText2TextPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print(snapshot)
pipe = T5SmallText2TextPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'source': pipe.source, 'dtype': 'float32'})

## 5. Generate and read the outputs correctly

`generate(text, max_new_tokens=..., num_beams=...)` runs one prefixed input through the encoder-decoder and returns a dict: `text` (the decoded generation with special tokens removed), `generated_tokens` (decoder tokens emitted, excluding `</s>`/pad), `input_tokens` (encoder tokens including `</s>`, the number checked against `MAX_INPUT_TOKENS`), `stopped_by` (`eos` when the model ended the sequence itself, `max_new_tokens` when it hit the ceiling — such an output is cut mid-thought and should be re-run with a larger `GEN_MAX_NEW_TOKENS` before anyone reads it as a finished summary), `known_prefix` (which trained prefix the input started with, or `None`), the `generation` settings actually used (`max_new_tokens`, `num_beams`, `do_sample=False`, `decision_rule`), `device`, `source` and the model identity. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind** — the counts above are counts, not scores; greedy decoding is an implicit per-step argmax with no minimum-probability cut-off, so some token is always produced, and the pipeline ships no acceptance threshold on output quality. Whoever deploys it owns any acceptance rule, judged on their own references. The run is deterministic for a given input, settings, weights, device and library versions (no sampling, `model.eval()`, no seed needed); float32 kernel differences between CPU and CUDA can flip a near-tied token and change the rest of the sequence from that point, and beam search can differ from greedy.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**. Summarisation and translation are conventionally scored with ROUGE-1/2/L and BLEU (or chrF) against human **reference outputs** — a reference summary per document, a reference translation per sentence — over enough items to state a dispersion; the synthetic sample has no references, so **no metric is reported** and none is manufactured from a proxy such as length ratio or copy rate. A caller who needs a number must supply references for a held-out sample of their own inputs and a scorer, and exclude or re-run outputs whose `stopped_by` is `max_new_tokens`. The upstream per-task figures in the paper's Table 14 are upstream claims, not measured here. The only checks below are falsifiable plumbing checks — one result per input, every count within its ceiling, every default input recognised by its prefix — plus a per-call wall time measured on the runtime identified in Section 1 (the first call includes warm-up). Look for a short German sentence for `input00` and an English condensation for `input01`; whether they are *good* is exactly what no number here can tell you.

In [ ]:
import time

results = []
for item_id, text in zip(item_ids, texts):
    started = time.perf_counter()
    result = pipe.generate(text, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    elapsed = time.perf_counter() - started
    results.append({'id': item_id, 'input': text, 'seconds': round(elapsed, 3), **result})
    print(f"{item_id} [{result['known_prefix'] or 'no known prefix'}] {result['input_tokens']} -> {result['generated_tokens']} tokens, stopped_by={result['stopped_by']}, {elapsed:.2f} s")
    print(f"    {result['text']}")
checks = {
    'one_result_per_input': len(results) == len(texts),
    'generated_within_ceiling': all(r['generated_tokens'] <= GEN_MAX_NEW_TOKENS for r in results),
    'input_within_ceiling': all(r['input_tokens'] <= MAX_INPUT_TOKENS for r in results),
    'settings_echoed': all(r['generation']['max_new_tokens'] == GEN_MAX_NEW_TOKENS and r['generation']['num_beams'] == NUM_BEAMS and r['generation']['do_sample'] is False for r in results),
}
if not USE_BYOD:
    checks['every_default_input_has_known_prefix'] = all(r['known_prefix'] is not None for r in results)
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
print({'checks': checks, 'decision_rule': results[0]['generation']['decision_rule'], 'hit_token_ceiling': [r['id'] for r in results if r['stopped_by'] == 'max_new_tokens']})
metrics = {}
print('no metric is reported: the sample has no reference outputs, and the repository ships no metric helper; compute ROUGE/BLEU on your own referenced inputs')

## 6. Export the generations and provenance

Two files are written under `outputs/`: `outputs/t5_small_text2text_generations.csv` — one row per input with its identifier, the known prefix, the input text, the generated text, both token counts, `stopped_by` and the wall time, so every generation maps back to its input — and `outputs/t5_small_text2text_result.json`, which carries the same items plus the generation settings in force, the ceilings, the sanity checks, the empty metric block, the sample identity and digest, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv
import json

os.makedirs('outputs', exist_ok=True)
items = [
    {'id': r['id'], 'known_prefix': r['known_prefix'], 'input': r['input'], 'output': r['text'], 'input_tokens': r['input_tokens'], 'generated_tokens': r['generated_tokens'], 'stopped_by': r['stopped_by'], 'seconds': r['seconds']}
    for r in results
]
with open('outputs/t5_small_text2text_generations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(items[0]))
    writer.writeheader()
    writer.writerows(items)
payload = {
    'items': items,
    'generation': results[0]['generation'],
    'ceilings': ceilings,
    'sanity_checks': checks,
    'metrics': metrics,
    'sample': {'name': sample_name, 'kind': sample_kind, 'inputs': len(texts), 'text_sha256': sample_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
        'source': pipe.source,
    },
}
with open('outputs/t5_small_text2text_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The generated strings are the model's continuation of *your* prefixed input under greedy (or beam) decoding: fluent text that can add, drop or invert a fact, and the pipeline attaches no probability, confidence or quality score to it — `generated_tokens`, `input_tokens` and `stopped_by` are counts and flags, not evidence of correctness. On the synthetic sample the checks prove only that the input contract, the prefix convention, the verified snapshot load and the generation path work end to end; **no metric is reported** because none can be computed without reference outputs, and a real evaluation needs referenced inputs from your own domain, a ROUGE/BLEU-style scorer, and enough items to state a dispersion. An unknown prefix produces plausible-looking output rather than an error; inputs above `MAX_INPUT_TOKENS` are refused rather than cut; outputs that stop at `max_new_tokens` are truncated mid-thought. The pipeline exposes no instruction following, sampling, batching, or the upstream `task_specific_params` decoding settings. Decoding is deterministic on a fixed device and dtype, but CPU and CUDA float32 kernels can diverge on a near-tied token.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, summarisation or translation quality on any domain, a usable acceptance threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/t5-small/` and rerun Section 4. `ValueError: input is N tokens; ceiling is MAX_INPUT_TOKENS=512` in Section 5: split or shorten that BYOD line and rerun from Section 2. `stopped_by` equal to `max_new_tokens`: raise `GEN_MAX_NEW_TOKENS` (ceiling `MAX_NEW_TOKENS`) and rerun Section 5. Output that copies the input: the line has no trained prefix, or the passage is too short to summarise.

**Next experiments.** Set `NUM_BEAMS = 4` and compare the beam-search outputs with the greedy ones (the card-pass smoke found the short translation unchanged); translate the same sentence with the French and Romanian prefixes; hand the `summarize: ` input a passage you wrote a one-sentence reference summary for and score the output with a ROUGE implementation of your choice — the first step towards a real evaluation; run the same batch on a CUDA runtime and diff the outputs against the CPU run. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/google-t5/t5-small
- Upstream code: https://github.com/google-research/text-to-text-transfer-transformer
- Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (Raffel et al., JMLR 2020): https://arxiv.org/abs/1910.10683